# Exercise 5 — Max the Builder


## Task

Max Welfare bought a remaining stock of **5,000 nails**, **33 buckets of finish**, and **770 planks** for EUR 500. He can build only fences and gazebos. Determine how many of each product he should build to maximize profit.

| Parameter | Unit | Fence | Gazebo |
|---|---|---:|---:|
| Nails | no. | 350 | 550 |
| Finish | no. | 4.5 | 3.5 |
| Planks | no. | 70 | 140 |
| Profit | EUR | 2000 | 2500 |

Tasks:

1. Formulate the problem algebraically.
2. Translate it into JuMP and solve it.
3. Interpret the solution.

The EUR 500 purchase price is a fixed cost and is therefore subtracted from the objective in the supplied implementation.


## Reference implementation


In [1]:
using JuMP, HiGHS
using DataFrames

# create dataframe from input table
df = DataFrame(
    Inputs = ["Nails", "Finish", "Planks"],
    Fence = [350, 4.5, 70],
    Gazebo = [550, 3.5, 140],
    Max_Production = [5000, 33, 770],
)

inputs = df.Inputs # production inputs
products = names(df)[2:3] # products

# create dicts
av = Dict(df.Inputs .=> df.Max_Production)
cm = Dict(products .=> [2000, 2500])
f = Dict((i, p) => df[findfirst(x -> x == i, df.Inputs), p] for i in inputs, p in products)

# model
model = Model(HiGHS.Optimizer)
@variable(model, x[products] >= 0) # production variable for each product
@constraint(
    model,
    production_constraint[i=inputs],
    f[i, "Fence"]*x["Fence"] + f[i, "Gazebo"]*x["Gazebo"] <= av[i]
) # max production
@objective(model, Max, sum(x[p]*cm[p] for p in products)-500)
optimize!(model)
objective_value(model)
value.(x)
dual.(production_constraint)


Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: Apple Accelerate 
LP has 3 rows; 2 cols; 6 nonzeros
Coefficient ranges:
  Matrix  [4e+00, 6e+02]
  Cost    [2e+03, 2e+03]
  Bound   [0e+00, 0e+00]
  RHS     [3e+01, 5e+03]
Presolving model
3 rows, 2 cols, 6 nonzeros 0s
3 rows, 2 cols, 6 nonzeros 0s
Presolve reductions: rows 3(-0); columns 2(-0); nonzeros 6(-0) - Not reduced
Problem not reduced by presolve: solving the LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0    -1.0156240074e+02 Ph1: 3(6.19141); Du: 2(101.562) 0.0s
          3     1.7000000000e+04 Pr: 0(0) 0.0s

Model status        : Optimal
Simplex   iterations: 3
Objective value     :  1.7000000000e+04
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00


1-dimensional DenseAxisArray{Float64,1,...} with index sets:
    Dimension 1, ["Nails", "Finish", "Planks"]
And data, a 3-element Vector{Float64}:
   -0.0
 -272.72727272727275
  -11.03896103896104